# Group Convolution — Companion Notebook

**6.7970/8.750 Symmetry and its Application to Machine Learning**

This notebook follows the Group Convolution exercise section by section.
Use it to **prototype your code** and **test your implementations**
against the course library before submitting on the website.

Each section includes small tests you can use to check your work.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atomicarchitects/symm4ml-colabs/blob/main/group_conv_companion.ipynb)

## Setup

In [2]:
%%capture
!pip install torch
!pip install pymatgen
!pip install https://symm4ml.mit.edu/_static/symm4ml_s26/symm4ml/symm4ml_latest.zip

In [3]:
import numpy as np
import itertools
import torch
import torch.nn.functional as F

from symm4ml import groups, rep, vib_modes, grids, group_conv

### Reference data

These groups, representations, and test tensors are used throughout the exercise for testing.

In [4]:
# C4 group (cyclic group of order 4, 2D rotation matrices)
_subset_C4 = np.array([
    [[np.cos(np.pi / 2), -np.sin(np.pi / 2)],
     [np.sin(np.pi / 2),  np.cos(np.pi / 2)]],
])
C4 = groups.generate_group(_subset_C4)
C4 = C4[[3, 1, 0, 2]]  # reorder so E is first
C4_torch = torch.tensor(C4).to(torch.float32)
C4_table = groups.make_multiplication_table(C4)
C4_reg_rep_torch = torch.tensor(rep.regular_representation(C4_table)).to(torch.float32)

# D4 group (dihedral group of order 8, 2D rotation + mirror matrices)
_subset_D4 = np.array([
    [[-1., 0.], [0., 1.]],
    [[np.cos(np.pi / 2), -np.sin(np.pi / 2)],
     [np.sin(np.pi / 2),  np.cos(np.pi / 2)]],
])
D4 = groups.generate_group(_subset_D4)
D4 = np.array(D4[::-1])  # reorder so E is first
D4_torch = torch.tensor(D4).to(torch.float32)
D4_table = groups.make_multiplication_table(D4)
D4_reg_rep_torch = torch.tensor(rep.regular_representation(D4_table)).to(torch.float32)

# Test tensors for reg_rep_group_convolution
trial1_input = torch.arange(0, 4, dtype=torch.float32).reshape(1, 1, 4)
trial1_filter = torch.arange(4, 8, dtype=torch.float32).reshape(1, 1, 4)
trial2_input = torch.arange(3 * 8, dtype=torch.float32).reshape(1, 3, 8)
trial2_filter = torch.arange(2 * 3 * 8, dtype=torch.float32).reshape(2, 3, 8)

# Test tensors for image2D functions
test_image_1 = torch.arange(4 * len(D4) * 5 * 5, dtype=torch.float32).reshape(1, 4, len(D4), 5, 5) * 0.1
test_filter_1 = torch.arange(2 * 4 * len(D4) * 3 * 3, dtype=torch.float32).reshape(2, 4, len(D4), 3, 3) * 0.1
test_image_2 = torch.arange(2 * len(C4) * 7 * 7, dtype=torch.float32).reshape(2, 1, len(C4), 7, 7) * 0.1
test_filter_2 = torch.arange(4 * len(C4) * 3 * 3, dtype=torch.float32).reshape(4, 1, len(C4), 3, 3) * 0.1

print(f"C4: {len(C4)} elements")
print(f"D4: {len(D4)} elements")

C4: 4 elements
D4: 8 elements


---
## 1. `reg_rep_group_convolution(reg_rep, input, filter)`

Group convolution over the left regular representation. Using Einstein summation notation:

$$[f\star \psi](k) = \sum_{h\in G}f(kh) \psi(h)$$

$$[f \star \psi] (k) = \sum_{i\in G} \sum_{j \in G} f(i) \psi(j) D^{\text{reg}} (k)_{ij}$$

$$[f\star \psi]_{dk} = \sum_{i \in G} \sum_{j \in G} f_{ci} \psi_{dcj} D^{\text{reg}}_{kij}$$

$$[f \star \psi]_{zdk} = \sum_{i \in G} \sum_{j \in G} f_{zci}\, \psi_{dcj}\, D^{\text{reg}}_{kij}$$

$$[L_u f\star \psi](g) = L_u[f\star \psi](g)$$

Implement this with `torch.einsum`.

In [5]:
def reg_rep_group_convolution(reg_rep, input, filter):
    """Performs group convolution of inputs and filters over the regular representation
    Input:
        reg_rep: torch.Tensor of shape [|G|, |G|, |G|] of the left regular representation
        input: torch.Tensor of shape [batch, channel_in, reg_rep_in]
        filter: torch.Tensor of shape [channel_out, channel_in, reg_rep_filter]
    Output:
        output: torch.Tensor of shape [batch, channel_out, reg_rep_out]
    """
    conv = torch.einsum('zci,dcj,kij->zdk', [input, filter, reg_rep])
    return conv

In [6]:
# No small tests in group_conv.py for reg_rep_group_convolution.
# Supplementary comparison with course (not a course small test):
np.testing.assert_allclose(
    reg_rep_group_convolution(C4_reg_rep_torch, trial1_input, trial1_filter).numpy(),
    group_conv.reg_rep_group_convolution(C4_reg_rep_torch, trial1_input, trial1_filter).numpy(),
    atol=1e-5
)
np.testing.assert_allclose(
    reg_rep_group_convolution(D4_reg_rep_torch, trial2_input, trial2_filter).numpy(),
    group_conv.reg_rep_group_convolution(D4_reg_rep_torch, trial2_input, trial2_filter).numpy(),
    atol=1e-5
)
print("reg_rep_group_convolution comparison passed!")

reg_rep_group_convolution comparison passed!


---
## Grid Utilities

Two simple utilities for assigning coordinates and getting indices for tensor/grid entries.

### 2. `grid_coords(dims)`

Create a coordinate grid for a tensor of specified dimensions. Each coordinate is generated by taking a `linspace` from $-1$ to $1$ for each dimension, then forming the Cartesian product.

In [32]:
def grid_coords(dims):
    """Create a coordinate grid for a tensor of specified dimensions.

    Input:
        dims : (iterable of int) The shape of the tensor. Each element
            specifies the number of pointsalong that dimension.

    Output:
        np.array representing the tensor coordinates. The returned array
        has shape (dof, d), where dof = ∏ dims (the total number of points)
        and d = len(dims). Each coordinate is generated by taking a linspace
        from -1 to 1 for each dimension.

    Notes:
        This function uses itertools.product to construct the full grid
        of coordinates from the provided dimensions.

    Examples:
    >>> grid_coords([2, 3])
    array([-1., -1.],
          [-1.,  0.],
          [-1.,  1.],
          [ 1., -1.],
          [ 1.,  0.],
          [ 1.,  1.]])
    """
    linespaces = []
    for v in dims:
        linespaces.append(np.linspace(-1, 1, v))
    result = []
    for k in itertools.product(*linespaces):
        result.append(k)
    return result

In [36]:
np.linspace(0,2,3)

array([0., 1., 2.])

In [34]:
# No small tests in grids.py for grid_coords.
# Supplementary comparison with course (not a course small test):
np.testing.assert_allclose(
    grid_coords([2, 3]),
    grids.grid_coords([2, 3]),
    atol=1e-7
)
np.testing.assert_allclose(
    grid_coords([3, 3, 3]),
    grids.grid_coords([3, 3, 3]),
    atol=1e-7
)
print("grid_coords comparison passed!")

grid_coords comparison passed!


### 3. `grid_indices(dims)`

Create an index grid for a tensor of specified dimensions. Each index is generated with `arange` for each dimension, then forming the Cartesian product.

In [40]:
def grid_indices(dims):
    """Create a coordinate grid for a tensor of specified dimensions.

    Input:
        dims : (iterable of int) The shape of the tensor. Each element
        specifies the number of points along that dimension.

    Output:
        np.array representing the tensor indices such that they can be
        flattened. The returned array has shape (dof, d), where dof = ∏ dims
        (the total number of points) and d = len(dims). Each coordinate is
        generated with arange for each dimension.

    Notes:
        This function uses itertools.product to construct the full grid of
        indices from the provided dimensions.

    Examples:
    >>> grid_indices([2, 3])
    array([[0, 0],
           [0, 1],
           [0, 2],
           [1, 0],
           [1, 1],
           [1, 2]])
    """
    linespaces = []
    for v in dims:
        linespaces.append(np.linspace(0, v-1, v))
    result = []
    for k in itertools.product(*linespaces):
        result.append(k)
    return result

In [42]:
# No small tests in grids.py for grid_indices.
# Supplementary comparison with course (not a course small test):
np.testing.assert_array_equal(
    grid_indices([2, 3]),
    grids.grid_indices([2, 3])
)
np.testing.assert_array_equal(
    grid_indices([3, 3, 3]),
    grids.grid_indices([3, 3, 3])
)
print("grid_indices comparison passed!")

grid_indices comparison passed!


---
## 2D Image Group Convolution

For images, group convolution operates over both spatial (2D rotations/mirrors) and regular representations:

$$[f \star \psi]_{zdk}(\mathbf{x}) = \sum_{\mathbf{y}} \sum_{i \in G} \sum_{j \in G} f_{zci}(\mathbf{x} + k\mathbf{y})\,\psi_{dcj}(\mathbf{y})\,D^{\text{reg}}(k)_{ij}$$

Where $f$ is the input, $\psi$ is the filter.

We implement this in three steps: compute the permutation representation for kernel pixels, create a rotated filter bank, then perform the full convolution.

### 4. `image2D_permutation_representation(rep_2D, kernel_size)`

Compute the permutation representation for pixels of a 2D image kernel. This bridges NumPy (geometric coordinate computation) and PyTorch (convolution operations).

**Hints:**
- Use `grids.grid_coords` to get pixel coordinates, multiplied by `np.array([1, -1])` to match image conventions (top-left = min x, max y).
- Use `vib_modes.permutation_representation` to compute the permutation representation from coordinates and 2D group matrices.

In [12]:
def image2D_permutation_representation(rep_2D:torch.Tensor, kernel_size):
    """Creates the permutation representation for pixels of a 2D image kernel.
    Input:
        rep_2D: torch.Tensor of shape [|G|, 2, 2]
        kernel_size: list of [kernel_height, kernel_width]
    Output:
        perm_rep: torch.Tensor of shape [|G|, kh*kw, kh*kw]
    """
    coords = grids.grid_coords(kernel_size)*np.array([1,-1])
    perm_rep = vib_modes.permutation_representation(coords, rep_2D.numpy())
    # YOUR CODE HERE
    return torch.from_numpy(perm_rep)

In [13]:
# No small tests in group_conv.py for image2D_permutation_representation.
# Supplementary comparison with course (not a course small test):
np.testing.assert_allclose(
    image2D_permutation_representation(C4_torch, [3, 3]).numpy(),
    group_conv.image2D_permutation_representation(C4_torch, [3, 3]).numpy(),
    atol=1e-5
)
np.testing.assert_allclose(
    image2D_permutation_representation(D4_torch, [3, 3]).numpy(),
    group_conv.image2D_permutation_representation(D4_torch, [3, 3]).numpy(),
    atol=1e-5
)
print("image2D_permutation_representation comparison passed!")

image2D_permutation_representation comparison passed!


In [14]:
a = np.arange(9).reshape(3,3)
a

array([[0, 1, 2],
       [3, 4, 5],
       [6, 7, 8]])

In [15]:
import numpy as np
a =  np.arange(9).reshape(3,3)
b = np.arange(3)
c_a = np.matmul(a,b)
print(f"a shape = {a.shape}, b shape = {b.shape}, c_a shape = {c_a.shape}")
c_b = np.einsum('ij,j->i',a,b)
print(c_a)
print(c_b)

a shape = (3, 3), b shape = (3,), c_a shape = (3,)
[ 5 14 23]
[ 5 14 23]


### 5. `image2D_group_convolution_filter_bank(perm_rep, filter)`

Now that we have the permutation representation, creating the filter bank is a single einsum! The `perm_rep` ($\Gamma^{\text{pix}}$) acts on the flattened `[kh, kw]` spatial dimensions of the filter, and adds a new `[reg_rep_out]` axis spanning group elements. Since this function only uses PyTorch operations (einsum + reshape), it's fully differentiable.

Code an implementation for creating the rotated filter bank we need for 2D image group convolution.

Create a rotated filter bank using the permutation representation. The `perm_rep` acts on the flattened spatial dimensions of the filter and adds a new axis spanning group elements. This is a single `torch.einsum` plus a reshape.

$$ \psi^{\text{bank}}_{fdcgi} = \sum_{g\in G}\sum_{j=0}^{k_h,k_w} = \Gamma_{fij}^{\text{pix}} \psi_{dcgj} $$

In [24]:
def image2D_group_convolution_filter_bank(perm_rep, filter):
    """Creates rotated filter bank for 2D image convolution
    Input:
        perm_rep: torch.Tensor of shape [|G|, kh*kw, kh*kw]
        filter: torch.Tensor of shape [channel_out, channel_in, reg_rep_filter, kernel_height, kernel_width]
    Output:
        filter_bank: torch.Tensor of shape [reg_rep_out, channel_out, channel_in, reg_rep_filter, kernel_height, kernel_width]
    """
    # flatten each filter(d, c, g_2) from kh,kw to kj*kw,1
    d = filter.shape[0] # channel_out
    c = filter.shape[1] # channel_in
    g = filter.shape[2] # reg_rep_filter
    kh = filter.shape[3] # kernel_height
    kw = filter.shape[4] # kernel_width
    f= perm_rep.shape[0] # |G|
    filt_reshaped = torch.reshape(filter, (d, c, g, kh*kw))
    print(f"perm_rep: {perm_rep.shape}, filter: {filter.shape}, filter_reshaped: {filt_reshaped.shape}")
    filter_bank = torch.einsum('fij,dcgj->fdcgi',[perm_rep,filt_reshaped])
    print(f"filter_bank: {filter_bank.shape}")
    filter_bank = torch.reshape(filter_bank,(g,d,c,f,kh,kw))
    print(f"filter_bank: {filter_bank.shape}")
    return filter_bank

In [25]:
# No small tests in group_conv.py for image2D_group_convolution_filter_bank.
# Supplementary comparison with course (not a course small test):
_D4_perm_rep_3x3 = group_conv.image2D_permutation_representation(D4_torch, [3, 3])
_C4_perm_rep_3x3 = group_conv.image2D_permutation_representation(C4_torch, [3, 3])
print(f"dim should be {group_conv.image2D_group_convolution_filter_bank(_D4_perm_rep_3x3, test_filter_1).shape}")
np.testing.assert_allclose(
    image2D_group_convolution_filter_bank(_D4_perm_rep_3x3, test_filter_1).numpy(),
    group_conv.image2D_group_convolution_filter_bank(_D4_perm_rep_3x3, test_filter_1).numpy(),
    atol=1e-5
)
np.testing.assert_allclose(
    image2D_group_convolution_filter_bank(_C4_perm_rep_3x3, test_filter_2).numpy(),
    group_conv.image2D_group_convolution_filter_bank(_C4_perm_rep_3x3, test_filter_2).numpy(),
    atol=1e-5
)
print("image2D_group_convolution_filter_bank comparison passed!")

dim should be torch.Size([8, 2, 4, 8, 3, 3])
perm_rep: torch.Size([8, 9, 9]), filter: torch.Size([2, 4, 8, 3, 3]), filter_reshaped: torch.Size([2, 4, 8, 9])
filter_bank: torch.Size([8, 2, 4, 8, 9])
filter_bank: torch.Size([8, 2, 4, 8, 3, 3])
perm_rep: torch.Size([4, 9, 9]), filter: torch.Size([4, 1, 4, 3, 3]), filter_reshaped: torch.Size([4, 1, 4, 9])
filter_bank: torch.Size([4, 4, 1, 4, 9])
filter_bank: torch.Size([4, 4, 1, 4, 3, 3])
image2D_group_convolution_filter_bank comparison passed!


### 6. `image2D_group_convolution(perm_rep, reg_rep, input, filter)`

Put everything together: create the rotated filter bank, contract with the regular representation, then use `F.conv2d` with `padding=1` for the spatial convolution.

**Hints:**
- Call `image2D_group_convolution_filter_bank` to create the rotated filter bank.
- Use `torch.einsum` with the `reg_rep` to contract the regular representation indices.
- Reshape input and filter bank to merge channel/group dimensions for `F.conv2d`, then unflatten the output.

In [47]:
def image2D_group_convolution(perm_rep, reg_rep, input, filter):
    """Performs group convolution of inputs and filters over the regular representation
    Input:
        perm_rep: torch.Tensor of shape [|G|, kh*kw, kh*kw]
        reg_rep: torch.Tensor of shape [|G|, |G|, |G|] of the left regular representation
        input: torch.Tensor of shape [batch, channel_in, reg_rep_in, height, width]
        filter: torch.Tensor of shape [channel_out, channel_in, reg_rep_filter, kernel_height, kernel_width]
    Output:
        output: torch.Tensor of shape [batch, channel_out, reg_rep_out, height, width]
    """
    d = filter.shape[0] # channel_out
    c = filter.shape[1] # channel_in
    g = filter.shape[2] # reg_rep_filter
    kh = filter.shape[3] # kernel_height
    kw = filter.shape[4] # kernel_width
    f = perm_rep.shape[0] # |G| = i,j
    b = input.shape[0]
    h = input.shape[3]
    w = input.shape[4]
    filt_bank = image2D_group_convolution_filter_bank(perm_rep, filter)
    # tensor [g,d,c,f,kh,kw] to [f, i, j] to get [g, d, i, c, kh, kw]
    tens_half = torch.einsum('gij,gdcjhw->gdichw', [reg_rep, filt_bank])
    # reshape tens_half from [g, i, d, c, kh, kw] to [d, c, kh, kw]
    tens_half = torch.reshape(tens_half, (g*d, f*c, kh, kw))
    # reshape input from [b, c, f, h, w] to [b, cf, h, w]
    input_reshaped = torch.reshape(input, (b, c*f, h, w))
    tens_next = F.conv2d(input_reshaped, tens_half)
    print(tens_next.shape)
    tens_next = torch.reshape(tens_next, (b, c, f, h, w))
    return tens_next

In [48]:
# No small tests in group_conv.py for image2D_group_convolution.
# Supplementary comparison with course (not a course small test):
_D4_perm_rep_3x3 = group_conv.image2D_permutation_representation(D4_torch, [3, 3])
_C4_perm_rep_3x3 = group_conv.image2D_permutation_representation(C4_torch, [3, 3])
np.testing.assert_allclose(
    image2D_group_convolution(_D4_perm_rep_3x3, D4_reg_rep_torch, test_image_1, test_filter_1).detach().numpy(),
    group_conv.image2D_group_convolution(_D4_perm_rep_3x3, D4_reg_rep_torch, test_image_1, test_filter_1).detach().numpy(),
    atol=1e-3
)
np.testing.assert_allclose(
    image2D_group_convolution(_C4_perm_rep_3x3, C4_reg_rep_torch, test_image_2, test_filter_2).detach().numpy(),
    group_conv.image2D_group_convolution(_C4_perm_rep_3x3, C4_reg_rep_torch, test_image_2, test_filter_2).detach().numpy(),
    atol=1e-3
)
print("image2D_group_convolution comparison passed!")

perm_rep: torch.Size([8, 9, 9]), filter: torch.Size([2, 4, 8, 3, 3]), filter_reshaped: torch.Size([2, 4, 8, 9])
filter_bank: torch.Size([8, 2, 4, 8, 9])
filter_bank: torch.Size([8, 2, 4, 8, 3, 3])
torch.Size([1, 16, 3, 3])


RuntimeError: shape '[1, 4, 8, 5, 5]' is invalid for input of size 144

---
## Explore Further

In [ ]:
# Try a different group or kernel size!

In [ ]:
# Experiment with group equivariance:
# Verify that L_u [f * psi] == [L_u f] * psi for some group element u